**Nama Anggota:**\
**1. Ammalya Khoirun Nizak (35031554036)**\
**2. Muhammad Rifky Ihwan (25031554241)**

# **Deskripsi Proyek**

Pada tugas sebelumnya (Feature Engineering), dataset yang dipakai berisi 318 judul berita yang telah melalui tahap preprocessing dan disimpan di kolom prepro.

# **Fokus proyek**

Mencari tahu seberapa mirip satu judul berita dengan judul lainnya. Melalui:
1.   Cosine Similariry (membandingkan vektor TF-IDF untuk melihat kesamaan arah antar teks).
2.   Jaccard Similarity (membandingkan kata dalam bentuk biner, lalu menghitung seberapa besar irisan kata dibandingan gabungannya.


# **Tujuan proyek**

Untuk menjawab pertanyaan judul berita mana yang paling mirip, seberapa kuat kemiripannya, dan apa perbedaan hasil COsine dengan Jaccard

# **Cosine Similarity**

Digunakkan untuk mengukur seberapa mirip dua teks dengan melihat sudut antara vektornya

*   Nilai mendekati 1 : sangat mirip.
*   Nilai mendekati 0 : hampir tidak mirip.
*   Karena TF-IDF menghasilkan angka non-negatif, maka skor Cosine Similarity selalu berada di rentang 0 sampai 1.





In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
df = pd.read_csv('antara_sepakbola_preprocessed.csv')

df = df.dropna().reset_index(drop=True)

df_pre = df['prepro']

print("Jumlah data :", len(df_pre))
df_pre.head()

Jumlah data : 318


,prepro
0,justin hubner mengaku keluarganya dapat ancama...
1,john herdman berharap temukan alphonso davies ...
2,justin hubner tak bisa gambarkan perasaannya s...
3,trofi fifa asean cup sangat berarti bagi ole r...
4,verdonk cs diminta herdman sebelum adu penalti...


### Feature Engineering dengan TF-IDF

Hasil representasi TF-IDF pada latihan sebelumnya yang akan digunakan sebagai input untuk menghitung Cosine Similarity.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_pre)

print("Ukuran matriks TF-IDF :", X.shape)

Ukuran matriks TF-IDF : (318, 998)


Hasilnya berupa matriks similarity, di mana setiap baris dan kolom merepresentasikan satu judul berita.
Nilai yang semakin mendekati 1 menunjukkan bahwa dua judul semakin mirip.

In [ ]:
similarity = cosine_similarity(X)

print("Ukuran matriks Cosine Similarity :", similarity.shape)

similarity[:10, :10]

Ukuran matriks Cosine Similarity : (318, 318)


array([[1.        , 0.        , 0.24174201, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 1.        , 0.        , 0.        , 0.04975923,
        0.        , 0.        , 0.        , 0.01872873, 0.        ],
       [0.24174201, 0.        , 1.        , 0.        , 0.        ,
        0.05842886, 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 1.        , 0.        ,
        0.        , 0.        , 0.11529465, 0.        , 0.08201284],
       [0.        , 0.04975923, 0.        , 0.        , 1.        ,
        0.13144199, 0.        , 0.        , 0.12763844, 0.        ],
       [0.        , 0.        , 0.05842886, 0.        , 0.13144199,
        1.        , 0.        , 0.        , 0.        , 0.12624373],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 1.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.       

Untuk mendapatkan hasil yang paling jelas, kita fokus mencari pasangan judul berita dengan nilai Cosine Similarity tertinggi.

* Bagian diagonal matriks tidak dipakai,
karena setiap dokumen otomatis memiliki nilai similarity = 1 terhadap dirinya sendiri.

* Jadi, yang kita cari adalah nilai tertinggi di luar diagonal, yaitu pasangan judul berbeda yang paling mirip menurut Cosine Similarity.

In [ ]:
pairs = []

for i in range(len(similarity)):
    for j in range(i + 1, len(similarity)):
        pairs.append((i, j, similarity[i, j]))

similarity_df = pd.DataFrame(pairs,
                             columns=['Document 1', 'Document 2', 'Cosine Similarity'])

similarity_df = similarity_df.sort_values(by='Cosine Similarity',
                                          ascending=False).reset_index(drop=True)

similarity_df.head(10)

,Document 1,Document 2,Cosine Similarity
0,51,60,0.695070
1,44,150,0.677915
2,285,289,0.667047
3,11,78,0.649349
4,287,297,0.622772
5,39,44,0.612421
6,19,20,0.594237
7,53,147,0.593442
8,6,38,0.591177
9,35,44,0.588798


In [ ]:
top_pairs = similarity_df.head(10).copy()

top_pairs['Text 1'] = top_pairs['Document 1'].apply(lambda x: df_pre.iloc[x])

top_pairs['Text 2'] = top_pairs['Document 2'].apply(lambda x: df_pre.iloc[x])

top_pairs[['Text 1', 'Text 2', 'Cosine Similarity']]

,Text 1,Text 2,Cosine Similarity
0,pelatih thailand sangat antusias hadapi indone...,final hadapi thailand john herdman mengaku san...,0.695070
1,indonesia juara fifa asean cup,pssi targetkan timnas indonesia juara fifa ase...,0.677915
2,akademi persib bandung juara hsl allstars usai...,putri garut tantang akademi persib bandung di ...,0.667047
3,tiket habis terjual final indonesia vs thaila...,tiket indonesia vs malaysia terjual gbk berpo...,0.649349
4,goal aksis juara hsl allstars,goal aksis melaju ke final u hsl,0.622772
5,presiden prabowo bangga indonesia juara fifa a...,indonesia juara fifa asean cup,0.612421
6,malaysia tempati peringkat ketiga fifa asean cup,malaysia raih peringkat ketiga fifa asean cup ...,0.594237
7,timnas putri u targetkan juara grup b kualifik...,indonesia berada di grup a kualifikasi piala a...,0.593442
8,erick thohir apresiasi perjuangan pemain timnas,erick thohir apresiasi timnas indonesia usai j...,0.591177
9,momen perayaan indonesia juara fifa asean cup,indonesia juara fifa asean cup,0.588798



## **Hasil Analisis Cosine Similarity**

* **Kemiripan tertinggi hanya 0.695** (dokumen 51 & 60), dan pasangan ke-10 masih 0.589. Tidak ada pasangan yang mendekati 1, artinya tidak ada dua judul yang identik setelah preprocessing. Kemiripan terkuat di dataset ini tergolong sedang hingga cukup kuat.
* **Pasangan teratas membahas peristiwa yang sama.** Dokumen 51 & 60 sama-sama memuat kata *thailand* dan *hadapi* (pembahasan laga melawan Thailand). Pola serupa terlihat pada berita tiket pertandingan (11 & 78: *tiket*, *indonesia*, *vs*, *terjual*), apresiasi Erick Thohir (6 & 38), dan Malaysia peringkat ketiga (19 & 20), yang hanya berbeda pada kata *tempati* dan *raih*.
* **Judul "indonesia juara fifa asean cup" (dokumen 44) menjadi pusat kemiripan.** Judul ini muncul di 3 dari 10 pasangan teratas, yaitu dengan dokumen 150, 39, dan 35, semuanya berita seputar juara FIFA ASEAN Cup.
* **Bobot TF-IDF membuat kata khas lebih berpengaruh.** Pasangan 287 & 297 hanya berbagi tiga kata (*goal*, *aksis*, *hsl*) tetapi masuk 5 besar, karena kata-kata khusus seperti ini cenderung berbobot lebih besar daripada kata yang umum di korpus seperti *indonesia* atau *juara*.
* **Kemiripan tinggi tidak selalu berarti makna yang sama.** Pasangan 53 & 147 berbagi kata *grup*, *kualifikasi*, dan *piala*, padahal judulnya menyebut grup yang berbeda (grup B dan grup A). Huruf grup bahkan tidak ikut dihitung karena token satu huruf dibuang oleh tokenizer bawaan. Cosine hanya membandingkan kata, bukan makna.

# **JACCARD**

Jaccard Similarity menghitung kemiripan dengan membandingkan jumlah kata yang sama dengan total kata unik dari dua dokumen.

* Untuk teks, biasanya digunakan Binary Bag of Words.

* Berbeda dengan TF-IDF, representasi biner hanya melihat apakah kata muncul atau tidak, tanpa memperhatikan bobot atau frekuensinya.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import jaccard_score

In [ ]:
binary_vectorizer = CountVectorizer(binary=True)

X_binary = binary_vectorizer.fit_transform(df_pre).toarray()

print("Ukuran matriks Binary BoW :", X_binary.shape)

Ukuran matriks Binary BoW : (318, 998)


Jaccard dihitung berdasarkan irisan dan gabungan fitur dari dua dokumen.

* Nilai mendekati 1 berarti dua dokumen memiliki banyak kata yang sama.
* Nilai mendekati 0 menunjukkan sedikit atau tidak ada kata yang sama.

In [ ]:
# Menghitung Jaccard Similarity
jaccard_pairs = []

for i in range(len(X_binary)):
    for j in range(i + 1, len(X_binary)):
        score = jaccard_score(X_binary[i],
                              X_binary[j])

        jaccard_pairs.append((i, j, score))

jaccard_df = pd.DataFrame(jaccard_pairs,
                          columns=['Document 1', 'Document 2', 'Jaccard Similarity'])

jaccard_df = jaccard_df.sort_values(by='Jaccard Similarity',
                                    ascending=False).reset_index(drop=True)

jaccard_df.head(10)

,Document 1,Document 2,Jaccard Similarity
0,35,44,0.714286
1,34,44,0.625000
2,39,44,0.625000
3,44,150,0.625000
4,44,110,0.555556
5,40,44,0.555556
6,110,150,0.545455
7,210,214,0.545455
8,19,20,0.545455
9,285,289,0.538462


In [ ]:
top_jaccard = jaccard_df.head(10).copy()

top_jaccard['Text 1'] = top_jaccard['Document 1'].apply(lambda x: df_pre.iloc[x])

top_jaccard['Text 2'] = top_jaccard['Document 2'].apply(lambda x: df_pre.iloc[x])

top_jaccard[['Text 1', 'Text 2', 'Jaccard Similarity']]

,Text 1,Text 2,Jaccard Similarity
0,momen perayaan indonesia juara fifa asean cup,indonesia juara fifa asean cup,0.714286
1,indonesia merayakan kemenangan jadi juara fifa...,indonesia juara fifa asean cup,0.625000
2,presiden prabowo bangga indonesia juara fifa a...,indonesia juara fifa asean cup,0.625000
3,indonesia juara fifa asean cup,pssi targetkan timnas indonesia juara fifa ase...,0.625000
4,indonesia juara fifa asean cup,dua legenda timnas berharap indonesia juara fi...,0.555556
5,bungkam thailand adu pinalti indonesia juara f...,indonesia juara fifa asean cup,0.555556
6,dua legenda timnas berharap indonesia juara fi...,pssi targetkan timnas indonesia juara fifa ase...,0.545455
7,persebaya tak gentar hadapi persib di final pi...,persebaya jumpa persib bandung di final piala ...,0.545455
8,malaysia tempati peringkat ketiga fifa asean cup,malaysia raih peringkat ketiga fifa asean cup ...,0.545455
9,akademi persib bandung juara hsl allstars usai...,putri garut tantang akademi persib bandung di ...,0.538462


## Analisis Hasil

Berdasarkan hasil Cosine Similarity dan Jaccard Similarity, kita bisa menemukan pasangan judul berita yang paling mirip.

* Cosine Similarity memakai nilai TF-IDF, sehingga mempertimbangkan bobot kata dalam dokumen.

* Jaccard Similarity memakai representasi biner, sehingga hanya melihat ada atau tidaknya kata yang sama.

* Perbedaan skor similarity muncul karena cara representasi fitur yang dipakai berbeda: Cosine fokus pada bobot kata, sedangkan Jaccard fokus pada keberadaan kata.

In [ ]:
print("Top 5 Cosine Similarity:")
display(top_pairs[['Text 1', 'Text 2', 'Cosine Similarity']].head(5))

print("\nTop 5 Jaccard Similarity:")
display(top_jaccard[['Text 1', 'Text 2', 'Jaccard Similarity']].head(5))

Top 5 Cosine Similarity:


,Text 1,Text 2,Cosine Similarity
0,pelatih thailand sangat antusias hadapi indone...,final hadapi thailand john herdman mengaku san...,0.695070
1,indonesia juara fifa asean cup,pssi targetkan timnas indonesia juara fifa ase...,0.677915
2,akademi persib bandung juara hsl allstars usai...,putri garut tantang akademi persib bandung di ...,0.667047
3,tiket habis terjual final indonesia vs thaila...,tiket indonesia vs malaysia terjual gbk berpo...,0.649349
4,goal aksis juara hsl allstars,goal aksis melaju ke final u hsl,0.622772



Top 5 Jaccard Similarity:


,Text 1,Text 2,Jaccard Similarity
0,momen perayaan indonesia juara fifa asean cup,indonesia juara fifa asean cup,0.714286
1,indonesia merayakan kemenangan jadi juara fifa...,indonesia juara fifa asean cup,0.625000
2,presiden prabowo bangga indonesia juara fifa a...,indonesia juara fifa asean cup,0.625000
3,indonesia juara fifa asean cup,pssi targetkan timnas indonesia juara fifa ase...,0.625000
4,indonesia juara fifa asean cup,dua legenda timnas berharap indonesia juara fi...,0.555556
